In [1]:
import pandas as pd
xls = pd.ExcelFile("../data/Supply chain logistics problem.xlsx", engine="openpyxl")
print(xls.sheet_names)


['OrderList', 'FreightRates', 'WhCosts', 'WhCapacities', 'ProductsPerPlant', 'VmiCustomers', 'PlantPorts']


In [5]:
wh_costs = pd.read_excel(xls, sheet_name = "WhCosts")
print(wh_costs.head())

wh_capacities = pd.read_excel(xls, sheet_name = "WhCapacities")
print(wh_capacities.head())

freight_rates = pd.read_excel(xls, sheet_name = "FreightRates")
print(freight_rates.head())

plant_ports = pd.read_excel(xls, sheet_name = "PlantPorts")
print(plant_ports.head())

        WH  Cost/unit
0  PLANT15   1.415063
1  PLANT17   0.428947
2  PLANT18   2.036254
3  PLANT05   0.488144
4  PLANT02   0.477504
  Plant ID  Daily Capacity 
0  PLANT15               11
1  PLANT17                8
2  PLANT18              111
3  PLANT05              385
4  PLANT02              138
  Carrier orig_port_cd dest_port_cd  minm_wgh_qty  max_wgh_qty svc_cd  \
0  V444_6       PORT08       PORT09         250.0       499.99    DTD   
1  V444_6       PORT08       PORT09          65.0        69.99    DTD   
2  V444_6       PORT08       PORT09          60.0        64.99    DTD   
3  V444_6       PORT08       PORT09          50.0        54.99    DTD   
4  V444_6       PORT08       PORT09          35.0        39.99    DTD   

   minimum cost    rate mode_dsc  tpt_day_cnt Carrier type  
0       43.2272  0.7132   AIR               2  V88888888_0  
1       43.2272  0.7512   AIR               2  V88888888_0  
2       43.2272  0.7892   AIR               2  V88888888_0  
3       43.2272  

In [10]:
order_list = pd.read_excel(xls, sheet_name = "OrderList")
print(order_list.head())
print(order_list.shape)

demand_by_port = order_list.groupby("Destination Port")["Unit quantity"].sum().reset_index()
print(demand_by_port)

       Order ID Order Date Origin Port Carrier  TPT Service Level  \
0  1.447296e+09 2013-05-26      PORT09   V44_3    1           CRF   
1  1.447158e+09 2013-05-26      PORT09   V44_3    1           CRF   
2  1.447139e+09 2013-05-26      PORT09   V44_3    1           CRF   
3  1.447364e+09 2013-05-26      PORT09   V44_3    1           CRF   
4  1.447364e+09 2013-05-26      PORT09   V44_3    1           CRF   

   Ship ahead day count  Ship Late Day count   Customer  Product ID  \
0                     3                    0  V55555_53     1700106   
1                     3                    0  V55555_53     1700106   
2                     3                    0  V55555_53     1700106   
3                     3                    0  V55555_53     1700106   
4                     3                    0  V55555_53     1700106   

  Plant Code Destination Port  Unit quantity  Weight  
0    PLANT16           PORT09            808   14.30  
1    PLANT16           PORT09           3188   8

In [13]:
print(order_list["Customer"].nunique())
demand_by_customer = order_list.groupby("Customer")["Unit quantity"].sum().reset_index()
print(demand_by_customer.head(10))

46
                 Customer  Unit quantity
0  V555555555555555555_17         266457
1  V555555555555555555_42         470632
2  V555555555555555555_45         116136
3  V555555555555555555_46          12080
4     V555555555555555_23            375
5     V555555555555555_29        1054980
6     V555555555555555_44          15125
7       V55555555555555_8         877824
8       V5555555555555_16           3434
9        V555555555555_31         435868


In [6]:
merged = order_list.merge(freight_rates, left_on = ["Carrier", "Origin Port", "Destination Port"], right_on = ["Carrier", "orig_port_cd", "dest_port_cd"], how = "left")
print(merged.head())
print(merged.shape)

print(order_list["Carrier"].unique()[:10])
print(freight_rates["Carrier"].unique()[:10])

print(order_list["Origin Port"].unique()[:10])
print(freight_rates["orig_port_cd"].unique()[:10])

matching_carriers = order_list["Carrier"].isin(["V444_0", "V444_1"])
print(matching_carriers.sum())

merged = order_list.merge(freight_rates, left_on = ["Carrier", "Origin Port", "Destination Port"], right_on = ["Carrier", "orig_port_cd", "dest_port_cd"], how = "inner")
print(merged.shape)

final = merged[(merged["Weight"] >= merged["minm_wgh_qty"]) & (merged["Weight"] <= merged["max_wgh_qty"])]
print(final.shape)

final_sorted = final.sort_values("rate")
final_dedup =  final_sorted.drop_duplicates(subset = "Order ID", keep = "first")
print(final_dedup.shape)
print(final_dedup.head())

NameError: name 'order_list' is not defined